# Model Forecasting Tutorial

The purpose of this notebook is to demonstrate reading and handling of spatial data and deploying a trained model on a spatial grid to generate predictions.

## Setup

In [ ]:
import xarray as xr
import rioxarray as rxr
from pyproj import Transformer
import tensorflow as tf
import numpy as np
import glob
import re
import os.path as osp
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
from utils import retrieve_url, str2time, read_pkl
from data_funcs import int2fstep
from moisture_rnn_xarray import bands_to_names, names_to_bands, get_file_list, preprocess, calc_eqs, calc_rain

In [ ]:
import importlib
import moisture_rnn_xarray
importlib.reload(moisture_rnn_xarray)
from moisture_rnn_xarray import bands_to_names, names_to_bands, get_file_list, preprocess

## User Options

In [ ]:
start_time = 2024042000            # start time for predictions (YYYMMDDHH)
end_time = 2024042002              # end time for predictions (YYYMMDDHH)
forecast_step = 3                  # forecast step for HRRR model
base_url = "https://demo.openwfm.org/web/data/fmda/tif/" # base URL for staged data
bbox = [37, -111, 46, -95]         # Spatial bounding box for preds (optional)
data_path = "data"

## Read in Model & Train Data

The fitted model and trained data determine which features should be read for predictions and how data scaling should be performed.

NOTE: run notebook `fmda_rnn_train_and_save.ipynb` to generate these files.

In [ ]:
# Training Data Object
rnn_dat = read_pkl("outputs/models/rnn_data_rocky.pkl")
# Fitted Model
mod = tf.keras.models.load_model("outputs/models/model_predict_raws_rocky.keras")

In [ ]:
mod.summary()

## Determine HRRR Data and Retrieve/Read

Based on user inputs and model specifications, generate a list of files needed to read in to create predictions. 

NOTE: this assumes a standard file structure and naming convention.

In [ ]:
print(f"Features used to train model: {rnn_dat.features_list}")

In [ ]:
bands = names_to_bands(rnn_dat.features_list)
print(f"Required HRRR Band Numbers: {bands}")
print(f"Required HRRR Band Names: {bands_to_names(bands)}")

### Retrieve Data Remotely

Based on the desired bands and forecast step, we create a list of files that are structured in such a way where `xarray` can easily read them and join by the proper dimensions. The dimensions of the resulting data need to include time and feature type, which we will refer to as "band" for now.

In order to calculate rain, we must read in the previous forecast step given by user input so we can calculate the hourly rainfall from the accumulated precipitation.

A small set of test data is staged on Demo, and we will retrieve it unless it already exists.

In [ ]:
fstep = int2fstep(forecast_step)
if forecast_step > 0:
    fprev = int2fstep(forecast_step-1)
print(f"{fstep=}")
print(f"{fprev=}")

In [ ]:
file_list = get_file_list(start_time, end_time, fstep=fstep, bands_list = [616, 620, 629])
file_list

In [ ]:
# Rain file list, used to calculate hourly rainfall
file_list_prev = get_file_list(start_time, end_time, fstep=fprev, bands_list = [629])
file_list_prev

In [ ]:
for sublist in file_list:
    for file in sublist:
        retrieve_url(
            f"{base_url}/{file}",
            dest_path = osp.join(data_path, file)
        )

In [ ]:
for sublist in file_list_prev:
    for file in sublist:
        print(file)
        retrieve_url(
            f"{base_url}/{file}",
            dest_path = osp.join(data_path, file)
        )

## Read Data

Using `xarray`, we read in the files and concatenate by time and feature type. This requires using a preprocessing function, which we named `preprocess` in the custom module. 

We read the data in and do some basic exploration...

In [ ]:
files = [[osp.join(data_path, filename) for filename in sublist] for sublist in file_list]
files_prev = [[osp.join(data_path, filename) for filename in sublist] for sublist in file_list_prev]

data = xr.open_mfdataset(
    files,
    concat_dim=["time", "band"],
    combine="nested",
    preprocess=preprocess
)

data.attrs["forecast_step"] = fstep

data_prev = xr.open_mfdataset(
    files_prev,
    concat_dim=["time", "band"],
    combine="nested",
    preprocess=preprocess
)

data_prev.attrs["forecast_step"] = fprev

In [ ]:
data

In [ ]:
print(data.dims)

In [ ]:
print(data.band_data.shape)

In [ ]:
plt.imshow(data.sel(band = "temp").isel(time=0).band_data)
plt.title(f"HRRR Temperature at Time 0")
# plt.colorbar(label="Temperature (°C)")
plt.show()

In [ ]:
data = calc_eqs(data)

In [ ]:
data = calc_rain(data, data_prev)

In [ ]:
data.band

In [ ]:
data.band_data.shape

In [ ]:
data.time[1]

In [ ]:
plt.imshow(data.sel(band = "rain", time = '2024-04-20T01:00:00.000000000')['band_data'])

In [ ]:
data

## Subsetting to bbox

In [ ]:
# def get_projection_info(ds, epsg = 4326):
#     # Given a geotiff file (a HRRR band), 
#     # return info necessary to transform lat/lon coords to the file structure
#     # Inputs: 
#     # ds: (osgeo.gdal.Dataset)
#     # epsg: (int) default 4326 for lon/lat
#     # Return: (tuple) with fields (ct, g_inv)
#         # ct: (osgeo.osr.CoordinateTransformation)
#         # gt_inv: (tuple) output of gdal.InvGeoTransform, also could be found with gdalinfo on command line
#     gt = ds.GetGeoTransform()
#     gp = ds.GetProjection()
#     if(ds.RasterCount>1):
#         print('Not Implemented for multiple Raster bands')
#         sys.exit(-1)
#     # Get Projection info
#     point_srs = osr.SpatialReference()
#     point_srs.ImportFromEPSG(4326) # hardcode for lon/lat
#     # GDAL>=3: make sure it's x/y
#     # see https://trac.osgeo.org/gdal/wiki/rfc73_proj6_wkt2_srsbarn
#     point_srs.SetAxisMappingStrategy(osr.OAMS_TRADITIONAL_GIS_ORDER)
#     file_srs = osr.SpatialReference()
#     file_srs.ImportFromWkt(gp)
#     ct = osr.CoordinateTransformation(point_srs, file_srs)
#     gt_inv = gdal.InvGeoTransform(gt)

#     return ct, gt_inv

In [ ]:
type(data)

In [ ]:
ds2 = rxr.open_rasterio(file_list[0][0])

In [ ]:
ds2.rio.transform()

In [ ]:
data.rio.crs

In [ ]:
ds2

In [ ]:
plt.imshow(ds2.isel(band=0))

In [ ]:
bbox = [37, -111, 46, -95]
crs = data.rio.crs

In [ ]:
def bbox_to_xy(bbox, crs, epsg = 4326):
    transformer = Transformer.from_crs(f"EPSG:{epsg}", crs, always_xy=True)
    # Transform the lat/lon bounding box to x/y
    minx, miny = transformer.transform(bbox[1], bbox[0])  # (min_lon, min_lat)
    maxx, maxy = transformer.transform(bbox[3], bbox[2])  # (max_lon, max_lat)

    return minx, miny, maxx, maxy

In [ ]:
minx, miny, maxx, maxy = bbox_to_xy(bbox, crs)

In [ ]:
ds2_clipped = ds2.rio.clip_box(minx=minx, miny=miny, maxx=maxx, maxy=maxy)

In [ ]:
plt.imshow(ds2_clipped.isel(band=0))

In [ ]:
ds2_clipped.coords

In [ ]:
features_list = ['Ed', 'Ew', 'rain']

In [ ]:
data_clipped = data.sel(x=slice(minx, maxx), y=slice(maxy, miny), band=features_list)  # Note: flip y for descending order

In [ ]:
data_clipped

In [ ]:
data_clipped.band_data.shape

In [ ]:
data_clipped.dims

In [ ]:
X = data_clipped.band_data.values

In [ ]:
X.shape

In [ ]:
type(X)

In [ ]:
X.shape

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [ ]:
scaler = StandardScaler()

In [ ]:
scaler

In [ ]:
data3 = data.sel(band = features_list)

In [ ]:
data3

In [ ]:
data3.band_data.shape

In [ ]:
data3.dims

In [ ]:
Xnew = data3.band_data.transpose("x", "y", "time", "band")

In [ ]:
Xnew.dims

In [ ]:
from utils import read_pkl
rnn_dat = read_pkl("../outputs/models/rnn_data_rocky.pkl")

In [ ]:
type(rnn_dat)

In [ ]:
rnn_dat.scaler

In [ ]:
rnn_dat.scaler.n_features_in_

In [ ]:
Xnew.x

In [ ]:
Xnew.isel(x=0,y=0,time=0,band=0).values

In [ ]:
Xnew.dims

In [ ]:
Xnew2 = Xnew.stack(spacetime=('x', 'y', 'time'))
Xnew2 = Xnew2.transpose('spacetime', 'band')
print(Xnew2.shape)

In [ ]:
# Xnew2 = rnn_dat.scaler.transform(Xnew2)

In [ ]:
# Xnew_scaled = xr.DataArray(Xnew2, dims=('spacetime', 'band'))


In [ ]:
# Transpose to (5715423, 3) for scaling
Xnew_stacked = Xnew2.transpose('spacetime', 'band')

# Apply scaling (resulting in a numpy array)
Xnew_scaled_array = rnn_dat.scaler.transform(Xnew_stacked)

# Convert scaled data back to a DataArray, using the same coords as Xnew_stacked
Xnew_scaled = xr.DataArray(Xnew_scaled_array, dims=('spacetime', 'band'), coords={'spacetime': Xnew_stacked['spacetime'], 'band': Xnew_stacked['band']})

# Unstack to return to original dimensions
Xnew_original = Xnew_scaled.unstack('spacetime')

In [ ]:
rnn_dat.features_list

In [ ]:
mod = tf.keras.models.load_model("../outputs/models/model_predict_raws_rocky.keras")

In [ ]:
mod.summary()